# 05 · Detection chips: a training dataset from the archive

SAR machine learning — ship, aircraft, and infrastructure detection; change
models; foundation-model pretraining — needs the archive cut into fixed-size,
georeferenced **image chips** with their metadata attached. `umbra chips` does
exactly that, and it does it *without downloading the scene*: it streams one
window at a time straight out of each acquisition's cloud-optimized GeoTIFF over
HTTP range reads, so memory stays bounded to a single tile no matter how large
the scene is.

This notebook chips one scene into a small dataset and inspects the manifest —
the record that makes each chip trainable: where it is on Earth, which pass it
came from, and the look-angle / resolution / polarization a model needs.

It needs the `load` extra (rasterio + numpy):

```bash
pip install "umbra-py[load]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Pick one scene to chip

A chip dataset is usually cut from *many* passes, but the mechanics are the
same for one — so we take a single GEC acquisition here to keep the notebook
fast and self-checking. `CHIPPABLE_ASSETS` names the products that carry a
chippable geocoded raster (`GEC`, the detected-amplitude COG, is the usual
choice).

In [ ]:
from umbra_py import CHIPPABLE_ASSETS, UmbraCatalog

print("chippable products:", CHIPPABLE_ASSETS)

# A one-day window keeps the search deterministic; take the first GEC scene.
scene = next(
    it
    for it in UmbraCatalog().search(
        start="2024-02-08", end="2024-02-08", product_types=["GEC"], limit=1
    )
    if "GEC" in it.available_assets
)
assert "GEC" in CHIPPABLE_ASSETS
print(scene.task, scene.datetime, "· resolution (m):", scene.resolution)

## 2 · Cut the scene into georeferenced chips

`write_chips` walks the scene's GEC one `chip_size` × `chip_size` window at a
time and writes each full-resolution tile as a GeoTIFF, alongside a manifest.
Two knobs keep the dataset clean:

- **`min_valid`** drops the mostly-nodata corners of a rotated footprint, so
  the dataset isn't padded with black squares.
- **`stride`** (unused here) would produce *overlapping* tiles for dense
  inference or augmentation.

Only each tile's bytes cross the network — there is no full-scene download.
We use a large `chip_size` so this one scene yields a modest, quick-to-write
dataset; a real run uses 256–512 px tiles over a whole search.

In [ ]:
import tempfile

from umbra_py import write_chips

out_dir = tempfile.mkdtemp(prefix="umbra-chips-")
dataset = write_chips(
    [scene],
    out_dir,
    asset="GEC",
    chip_size=2048,
    min_valid=0.5,
    manifest="manifest.jsonl",
)

assert dataset.records, "the scene produced no chips"
assert dataset.units == "amplitude"
print(f"{len(dataset.records)} chips written to {out_dir}")
print("manifest at", dataset.manifest_path)

## 3 · Read the manifest — what makes a chip trainable

The manifest is a `.jsonl` file, one JSON record per chip (the standard ML
format — stream it line by line into a `Dataset`). Every record is a pointer to
a real acquisition, never a guess: it carries the chip's geographic `bbox`,
`crs` and affine `transform`, its grid `row`/`col`, and the acquisition's
`datetime`, `place`, `polarizations`, `incidence_angle_deg` and resolution —
plus the mandatory CC-BY `attribution`, which travels with every chip.

In [ ]:
import json

from umbra_py import ATTRIBUTION

with open(dataset.manifest_path) as fh:
    records = [json.loads(line) for line in fh]

assert len(records) == len(dataset.records)

rec = records[0]
for key in ("item_id", "row", "col", "bbox", "crs", "polarizations", "incidence_angle_deg"):
    print(f"{key:20s} {rec[key]}")

# License propagation is non-negotiable — it must survive into the dataset.
assert all(r["attribution"] == ATTRIBUTION for r in records)
print("\nattribution:", rec["attribution"])

## 4 · Open one chip as a georeferenced array

Because each chip is a plain GeoTIFF, it opens with `rasterio` (or `rioxarray`,
or QGIS) and knows exactly where it sits on Earth — no sidecar needed. The
record's `path` is the chip's filename inside `out_dir` (the manifest lives
beside the chips), so we join it back to `out_dir` to read the pixels and
confirm the shape and CRS match the manifest.

In [ ]:
import os

import numpy as np
import rasterio

chip_path = os.path.join(out_dir, dataset.records[0].path)
with rasterio.open(chip_path) as ds:
    chip = ds.read(1)
    print("chip shape:", chip.shape, "· CRS:", ds.crs, "· dtype:", chip.dtype)

# A fixed size is a promise: partial edge tiles are dropped, so every chip is
# exactly chip_size square and ready to batch.
assert chip.shape == (2048, 2048)
finite = chip[np.isfinite(chip)]
assert finite.size, "chip has no valid pixels"
print(f"valid pixels: {finite.size / chip.size:.0%} · mean amplitude {finite.mean():.1f}")

## Where next

- **`.geojson` manifest** — pass `manifest="manifest.geojson"` to get a
  `FeatureCollection` of chip *footprints* instead, ready to drop into QGIS or
  geopandas to see the tiling over a basemap.
- **`umbra chips` on the CLI** mirrors this over a whole search (with
  `--local` / `--index-db` for the prebuilt index) — one command from catalog
  to training set.
- **`umbra convert`** turns a complex `SICD` into a geocoded COG first, so
  slant-plane products become chippable too.

*Contains Umbra open data, licensed under CC BY 4.0.*